# B7: Model Deployment as a Service

**Deliverable B Notebooks - Part B7**

This notebook documents the deployment of the requirement extraction model
as a FastAPI REST service.

## 1. API Endpoints

In [ ]:
endpoints = [
    {'method': 'POST', 'path': '/extract/text', 'desc': 'Extract from raw SRS text'},
    {'method': 'POST', 'path': '/extract/pdf', 'desc': 'Extract from uploaded PDF'},
    {'method': 'POST', 'path': '/assess/confidence', 'desc': 'Flag vague/incomplete requirements'},
    {'method': 'POST', 'path': '/assess/clarify', 'desc': 'Generate clarification questions'},
    {'method': 'POST', 'path': '/assess/integrate', 'desc': 'Integrate user answers'},
    {'method': 'GET',  'path': '/health', 'desc': 'Health check'},
]
print(f"{'Method':<8} {'Path':<30} {'Description'}")
print('-' * 70)
for ep in endpoints:
    print(f"{ep['method']:<8} {ep['path']:<30} {ep['desc']}")

## 2. Request/Response Schemas

In [ ]:
print('Request schema: /extract/text')
print('{')
print('  "document_name": "example-srs",')
print('  "text": "The system shall provide..."')
print('}')
print()
print('Response schema: ExtractionResult')
print('{')
print('  "document": "example-srs",')
print('  "requirements": [...],')
print('  "total_requirements": 25,')
print('  "functional": 18,')
print('  "non_functional": 7')
print('}')

## 3. Deployment

In [ ]:
# Development:
# cd ~/Final_Project/Requirement-extraction
# uv run uvicorn app.main:app --host 0.0.0.0 --port 8100 --reload

# Production:
# uv run uvicorn app.main:app --host 0.0.0.0 --port 8100
#     --workers 4 --timeout 300

# Docker:
# docker run -p 8100:8100 reqtracer:latest

print('Deployment commands documented above (commented)')

## 4. MLflow Model Packaging

In [ ]:
import mlflow

# Custom MLflow model wrapper
class MuseeExtractionModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.config = context.artifacts.get('model_config')

    def predict(self, context, model_input):
        # Call the FastAPI endpoint
        import requests
        response = requests.post(
            'http://localhost:8100/extract/text',
            json=model_input
        )
        return response.json()

print('Model wrapper class defined')
print('To package: mlflow.pyfunc.log_model(...)')

## 5. Validation Examples

In [ ]:
# Example validation (run against live server)
# import requests
# resp = requests.get('http://localhost:8100/health')
# assert resp.status_code == 200
# print('Health check passed')

# resp = requests.post('http://localhost:8100/extract/text', json={
#     'document_name': 'test',
#     'text': 'The system shall allow users to log in.'
# })
# assert resp.status_code == 200
# result = resp.json()
# assert 'requirements' in result
# print(f'Extracted {result["total_requirements"]} requirements')

print('Validation examples documented above (commented)')

## 6. Scalability

### Horizontal Scaling

- Run multiple uvicorn workers: `--workers 4`
- Use a load balancer (nginx) to distribute requests
- Each worker handles one document at a time

### Batching

- Process documents in parallel via asyncio.gather
- Each document split into 16K char chunks processed concurrently
- Weighted combination merge reconciles overlapping extractions

### Caching

- Cache extraction results for repeated documents (hash the input text)
- In-memory cache for dev, Redis for production

### Rate Limiting

- Nous API has rate limits
- Implement slowapi for request throttling
- Use exponential backoff on 429 responses

## 7. Model Artifacts

In [ ]:
import json

config_path = '/Users/xd/Final_Project/final-project-deliverables/model-artifacts/model_config.json'
with open(config_path) as f:
    cfg = json.load(f)

print('Model Configuration:')
for k, v in cfg.items():
    print(f'  {k}: {v}')